Author: Eva Rombouts  
Date: January 2024  

This notebook provides the code for training a text classification model using the Bert or Roberta model for Dutch language texts. It involves data loading, preprocessing, model training, and saving the trained model for future use. The focus is on predicting agitation scores based on client notes of nursing home residents.

Depending on the size of the dataset, it can be run on a CPU. However, using a GPU (here we use google CoLab) will drastically reduce training time.

In [ ]:
# Installations needed on Google CoLab
! pip install -U accelerate
! pip install -U transformers

In [ ]:
import pandas as pd
from google.colab import drive

from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, RobertaTokenizer, BertForSequenceClassification, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset, DataLoader
import torch

import os

In [ ]:
# Loading models and tokenizers for Dutch language models.
# Choose one by commenting out the other

# # (BERT model)
# # Initialize the tokenizer 
# tokenizer = BertTokenizer.from_pretrained('wietsedv/bert-base-dutch-cased')
# # Loading the pre-trained model with a classification head. 
# model = BertForSequenceClassification.from_pretrained('wietsedv/bert-base-dutch-cased', num_labels=2)
# # Path for saving the trained model
# model_save_path = '/content/drive/MyDrive/eColab/GenCareAI/models/BertClassification'


# (RoBERTa model)
# Initialize the tokenizer 
tokenizer = RobertaTokenizer.from_pretrained('pdelobelle/robbert-v2-dutch-base')
# Loading the pre-trained model with a classification head. 
model = RobertaForSequenceClassification.from_pretrained('pdelobelle/robbert-v2-dutch-base', num_labels=2)
# Path for saving the trained model
model_save_path = '/content/drive/MyDrive/eColab/GenCareAI/models/RobertaClassification'

In [ ]:
# Mount Google Drive to access the dataset
drive.mount('/content/drive')
# Load training, validation, and test datasets from Google Drive
df_train = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/df_train.csv')
df_valid = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/df_valid.csv')
df_test = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/df_test.csv')

In [ ]:
# Extract text and labels from the datasets
train_texts, train_labels = df_train['rapportage'].tolist(), df_train['onrust'].astype(int).tolist()
valid_texts, valid_labels = df_valid['rapportage'].tolist(), df_valid['onrust'].astype(int).tolist()
test_texts, test_labels = df_test['rapportage'].tolist(), df_test['onrust'].astype(int).tolist()

In [ ]:
# Create a custom dataset class for handling the data
class AgitationDataset(Dataset):
    def __init__(self, texts, labels):
        # Tokenize the texts
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=512)
        self.labels = labels

    def __getitem__(self, idx):
        # Return a single tokenized item and its label
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        # Return the total number of items
        return len(self.labels)


In [ ]:
# Create datasets for training and validation
train_dataset = AgitationDataset(train_texts, train_labels)
test_dataset = AgitationDataset(valid_texts, valid_labels)

In [ ]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

In [ ]:
# Initialize the Trainer with the model, training arguments, and datasets
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset            # evaluation dataset
)

In [ ]:
# Train the model
trainer.train()

In [ ]:
# Create the directory if it doesn't exist
os.makedirs(model_save_path, exist_ok=True)
# Save trained model
model.save_pretrained(model_save_path)